# 🗺️ მოსახლეობის სიმჭიდროვის გამოთვლა და ვიზუალიზაცია

**ipyleaflet + GeoPandas | Google Colab**

---

ეს notebook ასრულებს **PyQGIS**-ის ანალოგიურ ამოცანას — მოსახლეობის სიმჭიდროვის გამოთვლას — სუფთა Python გარემოში, Google Colab-ზე.

## 📐 ფორმულა

$$D = \frac{P}{A}$$

სადაც:
- **D** — მოსახლეობის სიმჭიდროვე (ად/კვ.კმ)
- **P** — მოსახლეობის რაოდენობა (`Mosaxl_200`)
- **A** — ფართობი კვ. კილომეტრში (`area_km`)

---

## 🔧 გამოყენებული ბიბლიოთეკები

| ბიბლიოთეკა | მიზანი |
|---|---|
| `geopandas` | სივრცითი მონაცემების დამუშავება (QGIS-ის ანალოგი) |
| `ipyleaflet` | ინტერაქტიული რუკა Jupyter/Colab-ში |
| `branca` | ფერის სკალა (colormap) ლეგენდისთვის |
| `pandas` | ატრიბუტების ცხრილი |
| `numpy` | რიცხვითი გამოთვლები |

## 📦 ნაბიჯი 1 — ბიბლიოთეკების ინსტალაცია

> Google Colab-ზე `geopandas` და `ipyleaflet` წინასწარ დაყენება სჭირდება.

In [ ]:
# ბიბლიოთეკების ინსტალაცია Google Colab-ზე
!pip install geopandas ipyleaflet branca --quiet

## 📥 ნაბიჯი 2 — ბიბლიოთეკების იმპორტი

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import json
import requests

import ipyleaflet
from ipyleaflet import (
    Map, GeoJSON, Choropleth, LegendControl,
    WidgetControl, LayersControl, basemaps
)
from ipywidgets import HTML, Output
import branca.colormap as cm

print(f"✅ geopandas  v{gpd.__version__}")
print(f"✅ ipyleaflet v{ipyleaflet.__version__}")
print(f"✅ pandas     v{pd.__version__}")

## 🗺️ ნაბიჯი 3 — საქართველოს მუნიციპალიტეტების მონაცემების ჩატვირთვა

ვიყენებთ **GADM** (Global Administrative Areas) ბაზის საქართველოს მე-2 დონის ადმინისტრაციული ერთეულების GeoJSON-ს.  
ამ მაგალითში სინთეტიკური (`Mosaxl_200`) მოსახლეობის მონაცემები ემატება, PyQGIS მაგალითის ანალოგიურად.

In [ ]:
# -------------------------------------------------------
# საქართველოს მუნიციპალიტეტების GeoJSON-ის ჩატვირთვა
# GADM v4.1 — Georgia, Admin Level 2 (მუნიციპალიტეტები)
# -------------------------------------------------------
GEOJSON_URL = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
    "master/geojson/ne_10m_admin_1_states_provinces.geojson"
)

# ჩამოტვირთვა და ფილტრაცია საქართველოსთვის
gdf_world = gpd.read_file(GEOJSON_URL)
gdf = gdf_world[gdf_world['admin'] == 'Georgia'].copy()
gdf = gdf.reset_index(drop=True)

print(f"📌 ჩატვირთული ობიექტების რაოდენობა: {len(gdf)}")
print(f"📌 CRS: {gdf.crs}")
gdf[['name', 'geometry']].head()

## 🔢 ნაბიჯი 4 — სინთეტიკური მოსახლეობის მონაცემების დამატება

PyQGIS მაგალითის ანალოგიური ველი `Mosaxl_200` — მოსახლეობის მონაცემი 2000-იანი წლებისთვის.  
რეალური პროექტისთვის ეს ველი GeoPackage/Shapefile-ში უნდა არსებობდეს.

In [ ]:
# -------------------------------------------------------
# სინთეტიკური მოსახლეობის მონაცემები (Mosaxl_200)
# საქართველოს რეგიონების სავარაუდო მოსახლეობა 2002 წ.
# -------------------------------------------------------
np.random.seed(42)

# რეალური საქართველოს რეგიონული მოსახლეობის სავარაუდო მნიშვნელობები
population_map = {
    "Tbilisi": 1081679,
    "Adjara": 376016,
    "Guria": 113350,
    "Imereti": 699666,
    "Kakheti": 407182,
    "Kvemo Kartli": 497531,
    "Mtskheta-Mtianeti": 125141,
    "Racha-Lechkhumi and Kvemo Svaneti": 50969,
    "Samegrelo-Zemo Svaneti": 466.100,
    "Samtskhe-Javakheti": 207598,
    "Shida Kartli": 314801,
    "Abkhazia": 215272,
    "South Ossetia": 70000
}

def assign_population(name):
    """მოსახლეობის მინიჭება სახელის მიხედვით, სხვა შემთხვევაში — შემთხვევითი"""
    for key, val in population_map.items():
        if key.lower() in str(name).lower() or str(name).lower() in key.lower():
            return int(val)
    return np.random.randint(50_000, 500_000)

gdf['Mosaxl_200'] = gdf['name'].apply(assign_population)

print("✅ Mosaxl_200 ველი დამატებულია")
gdf[['name', 'Mosaxl_200']].sort_values('Mosaxl_200', ascending=False)

## 📐 ნაბიჯი 5 — ფართობის გამოთვლა და მოსახლეობის სიმჭიდროვე

**PyQGIS-ის ანალოგი** — `$area`, `"area"/1000000`, `"Mosaxl_200"/area_km`:

```python
# QGIS-ში:
expr1 = QgsExpression('$area')                 # კვ. მეტრი
expr2 = QgsExpression('"area"/1000000')        # კვ. კმ
expr3 = QgsExpression('"Mosaxl_200"/area_km')  # სიმჭიდროვე

# GeoPandas-ში ეს ასე:
gdf['area']       = gdf.to_crs(epsg=32638).area          # კვ. მეტრი
gdf['area_km']    = gdf['area'] / 1_000_000               # კვ. კმ
gdf['simwidrove'] = gdf['Mosaxl_200'] / gdf['area_km']    # ად/კვ.კმ
```

In [ ]:
# -------------------------------------------------------
# ფართობის გამოთვლა
# EPSG:32638 — UTM Zone 38N (საქართველოსთვის შესაფერისი)
# -------------------------------------------------------

# საწყისი CRS შენახვა (WGS84 — ipyleaflet-ისთვის)
gdf_wgs84 = gdf.copy()

# ფართობის გამოთვლა პროექციულ სისტემაში (კვ. მეტრი)
gdf_projected = gdf.to_crs(epsg=32638)
gdf_wgs84['area'] = gdf_projected.area                         # expr1: $area
gdf_wgs84['area_km'] = gdf_wgs84['area'] / 1_000_000           # expr2: "area"/1000000
gdf_wgs84['simwidrove'] = (                                     # expr3: "Mosaxl_200"/area_km
    gdf_wgs84['Mosaxl_200'] / gdf_wgs84['area_km']
)

print("✅ ველები გამოთვლილია:")
print(f"   area       — ფართობი კვ. მეტრში")
print(f"   area_km    — ფართობი კვ. კილომეტრში")
print(f"   simwidrove — მოსახლეობის სიმჭიდროვე (ად/კვ.კმ)")
print()

# შედეგების ცხრილი
result_df = gdf_wgs84[['name', 'Mosaxl_200', 'area_km', 'simwidrove']].copy()
result_df['area_km'] = result_df['area_km'].round(2)
result_df['simwidrove'] = result_df['simwidrove'].round(2)
result_df.sort_values('simwidrove', ascending=False).reset_index(drop=True)

## 🗺️ ნაბიჯი 6 — ipyleaflet Choropleth რუკა

ვიზუალიზაცია **choropleth** ტიპის რუკით — სიმჭიდროვის მნიშვნელობა ფერის ინტენსივობით.

- 🟡 ღია ყვითელი → დაბალი სიმჭიდროვე  
- 🔴 მუქი წითელი → მაღალი სიმჭიდროვე  

> hover-ზე გამოჩნდება მუნიციპალიტეტის სახელი და სიმჭიდროვე.

In [ ]:
# -------------------------------------------------------
# GeoJSON მომზადება ipyleaflet-ისთვის
# -------------------------------------------------------

# GeoDataFrame-ის GeoJSON-ად გარდაქმნა
geojson_data = json.loads(gdf_wgs84.to_json())

# სიმჭიდროვის dictionary (Choropleth-ისთვის)
choro_data = dict(
    zip(
        gdf_wgs84['name'].astype(str),
        gdf_wgs84['simwidrove'].round(2)
    )
)

print(f"✅ GeoJSON მომზადებულია — {len(geojson_data['features'])} ობიექტი")
print("\n📊 სიმჭიდროვის მნიშვნელობები (ად/კვ.კმ):")
for name, val in sorted(choro_data.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * min(int(val / 50), 30)
    print(f"  {name:<40} {val:>8.1f}  {bar}")

In [ ]:
# -------------------------------------------------------
# ipyleaflet Choropleth რუკის შექმნა
# -------------------------------------------------------

# ფერის სკალა (YlOrRd — ყვითელი → ნარინჯისფერი → წითელი)
colormap = cm.LinearColormap(
    colors=['#ffffb2', '#fecc5c', '#fd8d3c', '#f03b20', '#bd0026'],
    vmin=gdf_wgs84['simwidrove'].min(),
    vmax=gdf_wgs84['simwidrove'].quantile(0.95),  # outlier-ის კლიპვა
    caption='მოსახლეობის სიმჭიდროვე (ად/კვ.კმ)'
)

# რუკის შექმნა — საქართველოს ცენტრი
m = Map(
    center=(42.0, 43.5),
    zoom=7,
    basemap=basemaps.CartoDB.Positron,
    scroll_wheel_zoom=True
)

# Choropleth შრე
choropleth = Choropleth(
    geo_data=geojson_data,
    choro_data=choro_data,
    colormap=colormap,
    key_on='feature.properties.name',
    border_color='white',
    style={
        'fillOpacity': 0.75,
        'weight': 1.5,
        'dashArray': '3'
    },
    hover_style={
        'fillOpacity': 0.95,
        'weight': 3,
        'dashArray': '0'
    },
    name='მოსახლეობის სიმჭიდროვე'
)

m.add_layer(choropleth)

# ფერის სკალის ლეგენდა
colormap.add_to(m)

# შრეების კონტროლი
m.add_control(LayersControl(position='topright'))

# სათაური
title_html = HTML(
    value="""
    <div style='
        background: rgba(255,255,255,0.92);
        padding: 10px 16px;
        border-radius: 8px;
        border-left: 4px solid #bd0026;
        font-family: sans-serif;
        box-shadow: 0 2px 8px rgba(0,0,0,0.15);
    '>
        <b style='font-size:14px;'>🗺️ საქართველო</b><br>
        <span style='font-size:12px; color:#555;'>მოსახლეობის სიმჭიდროვე — ად/კვ.კმ</span>
    </div>
    """
)
m.add_control(WidgetControl(widget=title_html, position='topleft'))

# Hover ინფო პანელი
info_html = HTML(
    value="<div style='font-family:sans-serif; font-size:12px; color:#777; padding:6px;'>hover რეგიონზე</div>"
)
info_widget = WidgetControl(widget=info_html, position='bottomleft')
m.add_control(info_widget)

def on_hover(event, feature, **kwargs):
    """Hover-ზე ინფოს ჩვენება"""
    name = feature['properties'].get('name', '?')
    density = choro_data.get(name, 0)
    area = gdf_wgs84.loc[gdf_wgs84['name'] == name, 'area_km'].values
    pop = gdf_wgs84.loc[gdf_wgs84['name'] == name, 'Mosaxl_200'].values
    area_str = f"{area[0]:,.0f}" if len(area) else "?"
    pop_str = f"{pop[0]:,}" if len(pop) else "?"
    info_html.value = f"""
    <div style='
        background: rgba(255,255,255,0.95);
        padding: 10px 14px;
        border-radius: 6px;
        font-family: sans-serif;
        font-size: 12px;
        border-left: 3px solid #bd0026;
        min-width: 180px;
    '>
        <b style='font-size:13px;'>{name}</b><br>
        👥 მოსახლეობა: <b>{pop_str}</b><br>
        📐 ფართობი: <b>{area_str} კვ.კმ</b><br>
        📊 სიმჭიდროვე: <b>{density:.1f} ად/კვ.კმ</b>
    </div>
    """

choropleth.on_hover(on_hover)

print("✅ რუკა მზადაა — ქვემოთ ნახავთ ვიზუალიზაციას")
m

## 📊 ნაბიჯი 7 — შედეგების ცხრილი

**PyQGIS** Attribute Table-ის ანალოგი — სრული გამოთვლილი ატრიბუტები.

In [ ]:
# -------------------------------------------------------
# შედეგების ცხრილი — Attribute Table (QGIS ანალოგი)
# -------------------------------------------------------
display_df = gdf_wgs84[[
    'name', 'Mosaxl_200', 'area', 'area_km', 'simwidrove'
]].copy()

display_df.columns = [
    'რეგიონი', 'მოსახლეობა (Mosaxl_200)',
    'ფართობი (კვ.მ)', 'ფართობი (კვ.კმ)', 'სიმჭიდროვე (ად/კვ.კმ)'
]

display_df['ფართობი (კვ.მ)'] = display_df['ფართობი (კვ.მ)'].apply(lambda x: f"{x:,.0f}")
display_df['ფართობი (კვ.კმ)'] = display_df['ფართობი (კვ.კმ)'].apply(lambda x: f"{x:,.2f}")
display_df['სიმჭიდროვე (ად/კვ.კმ)'] = display_df['სიმჭიდროვე (ად/კვ.კმ)'].apply(lambda x: f"{x:.2f}")
display_df['მოსახლეობა (Mosaxl_200)'] = display_df['მოსახლეობა (Mosaxl_200)'].apply(lambda x: f"{x:,}")

display_df.reset_index(drop=True, inplace=True)
display_df

## 💾 ნაბიჯი 8 — შედეგების შენახვა

გამოთვლილი მონაცემების ექსპორტი **GeoJSON** და **CSV** ფორმატებში.

In [ ]:
# -------------------------------------------------------
# შედეგების ექსპორტი
# -------------------------------------------------------

# GeoJSON შენახვა (QGIS-ში გახსნისთვის)
export_cols = ['name', 'Mosaxl_200', 'area', 'area_km', 'simwidrove', 'geometry']
gdf_export = gdf_wgs84[export_cols].copy()
gdf_export.to_file('municipalitys_density.geojson', driver='GeoJSON')
print("✅ GeoJSON შენახულია: municipalitys_density.geojson")

# CSV შენახვა (ატრიბუტების ცხრილი)
csv_cols = ['name', 'Mosaxl_200', 'area', 'area_km', 'simwidrove']
gdf_wgs84[csv_cols].to_csv('municipalitys_density.csv', index=False, encoding='utf-8-sig')
print("✅ CSV შენახულია: municipalitys_density.csv")

# Google Colab-ზე ჩამოტვირთვა
try:
    from google.colab import files
    files.download('municipalitys_density.csv')
    files.download('municipalitys_density.geojson')
    print("📥 ფაილები ჩამოიტვირთება...")
except ImportError:
    print("ℹ️  Colab გარემო არ არის — ფაილები მიმდინარე დირექტორიაში შეინახება")

---

## 📝 შეჯამება — PyQGIS vs GeoPandas შედარება

| ოპერაცია | PyQGIS | GeoPandas (Colab) |
|---|---|---|
| შრის გახსნა | `QgsProject.instance().mapLayersByName()` | `gpd.read_file()` |
| ველის დამატება | `pv.addAttributes([QgsField(...)])` | `gdf['ველი'] = ...` |
| ფართობი | `QgsExpression('$area')` | `gdf.to_crs(32638).area` |
| ფართობი კვ.კმ | `QgsExpression('"area"/1000000')` | `gdf['area'] / 1_000_000` |
| სიმჭიდროვე | `QgsExpression('"Mosaxl_200"/area_km')` | `gdf['Mosaxl_200'] / gdf['area_km']` |
| ატრიბუტების განახლება | `layer.updateFeature(i)` | ავტომატური (pandas) |
| ვიზუალიზაცია | QGIS Layer Styling | `ipyleaflet.Choropleth` |
| ექსპორტი | Save As... | `gdf.to_file()` / `df.to_csv()` |

---

> **📌 შენიშვნა:** რეალური პროექტისთვის `Mosaxl_200` ველი GeoPackage ან Shapefile-ში  
> უნდა იყოს — `gpd.read_file('Municipalitys.gpkg')` და ყველა სხვა ნაბიჯი  
> ზუსტად ასეა გამოყენებული.

**osdoc.qgis.ge** | GTU PyQGIS კურსი